# Load Airports

Load airports from `airline_routes.json` into a pandas DataFrame.

In [1]:
import pandas as pd
url = 'https://raw.githubusercontent.com/Jonty/airline-route-data/refs/heads/main/airline_routes.json'
routes = pd.read_json(url).transpose()
print(routes.shape)
routes.head(3)

/Users/jonzimmerman/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


(3854, 13)


,city_name,continent,country,country_code,display_name,elevation,iata,icao,latitude,longitude,name,routes,timezone
AAA,Anaa,OC,French Polynesia,PF,"Anaa (AAA), French Polynesia",23,AAA,NTGA,-17.355648,-145.50913,Anaa Airport,"[{'carriers': [{'iata': 'VT', 'name': 'Air Tah...",Pacific/Tahiti
AAD,Adado,AF,Somalia,SO,"Adado (AAD), Somalia",1005,AAD,None,6.095833,46.6375,Adado Airport,"[{'carriers': [], 'iata': 'MGQ', 'km': 474, 'm...",Africa/Mogadishu
AAE,Annaba,AF,Algeria,DZ,"Annaba (AAE), Algeria",16,AAE,DABB,36.821392,7.811857,Annaba,"[{'carriers': [{'iata': 'AH', 'name': 'Air Alg...",Africa/Algiers


In [2]:
import json

pdx = routes[routes['iata']=="PDX"]
routes_list = pdx["routes"].iloc[0]
pd.DataFrame(routes_list).head(3)

,carriers,iata,km,min
0,"[{'iata': 'AS', 'name': 'Alaska Airlines'}, {'...",SEA,207,65
1,"[{'iata': 'AA', 'name': 'American Airlines'}, ...",LAX,1341,149
2,"[{'iata': 'AA', 'name': 'American Airlines'}, ...",PHX,1624,159


In [3]:
dfs = []
for iata_code, row in routes.iterrows():
    routes_list = row["routes"]
    if not routes_list:
        continue
    df = pd.DataFrame(routes_list)
    df["dest_iata"] = iata_code
    dfs.append(df)
destinations = pd.concat(dfs, ignore_index=True)
destinations[destinations['iata']=="PDX"].head(3)

,carriers,iata,km,min,dest_iata
169,"[{'iata': 'AS', 'name': 'Alaska Airlines'}]",PDX,1786,183,ABQ
2186,"[{'iata': 'KL', 'name': 'KLM'}]",PDX,8052,605,AMS
2253,"[{'iata': 'AS', 'name': 'Alaska Airlines'}]",PDX,2482,220,ANC


In [4]:
master_air_df = pd.merge(
    routes,
    destinations,
    on = 'iata'
)

print(routes.shape)
print(destinations.shape)
print(master_air_df.shape)

master_air_df[master_air_df['iata']=="PDX"].head(3)

(3854, 13)
(58342, 5)
(58342, 17)


,city_name,continent,country,country_code,display_name,elevation,iata,icao,latitude,longitude,name,routes,timezone,carriers,km,min,dest_iata
39905,Portland,NA,USA,US,"Portland (PDX), USA",30,PDX,KPDX,45.588995,-122.592901,Portland International,"[{'carriers': [{'iata': 'AS', 'name': 'Alaska ...",America/Los_Angeles,"[{'iata': 'AS', 'name': 'Alaska Airlines'}]",1786,183,ABQ
39906,Portland,NA,USA,US,"Portland (PDX), USA",30,PDX,KPDX,45.588995,-122.592901,Portland International,"[{'carriers': [{'iata': 'AS', 'name': 'Alaska ...",America/Los_Angeles,"[{'iata': 'KL', 'name': 'KLM'}]",8052,605,AMS
39907,Portland,NA,USA,US,"Portland (PDX), USA",30,PDX,KPDX,45.588995,-122.592901,Portland International,"[{'carriers': [{'iata': 'AS', 'name': 'Alaska ...",America/Los_Angeles,"[{'iata': 'AS', 'name': 'Alaska Airlines'}]",2482,220,ANC


Let's do a self-join to get the name of the destination airport

In [5]:
source_name = master_air_df[['name','iata']].drop_duplicates()
source_name = source_name.rename(
    columns={
        'name':'dest_name',
        'iata':'dest_iata'
    }
)

print(master_air_df.shape)
master_air_df = pd.merge(
    master_air_df,
    source_name,
    on = 'dest_iata',
    how = 'left'
)

print(source_name.shape)
print(master_air_df.shape)

master_air_df[master_air_df['iata']=="PDX"].head(3)

(58342, 17)
(3808, 2)
(58342, 18)


,city_name,continent,country,country_code,display_name,elevation,iata,icao,latitude,longitude,name,routes,timezone,carriers,km,min,dest_iata,dest_name
39905,Portland,NA,USA,US,"Portland (PDX), USA",30,PDX,KPDX,45.588995,-122.592901,Portland International,"[{'carriers': [{'iata': 'AS', 'name': 'Alaska ...",America/Los_Angeles,"[{'iata': 'AS', 'name': 'Alaska Airlines'}]",1786,183,ABQ,Albuquerque International Sunport
39906,Portland,NA,USA,US,"Portland (PDX), USA",30,PDX,KPDX,45.588995,-122.592901,Portland International,"[{'carriers': [{'iata': 'AS', 'name': 'Alaska ...",America/Los_Angeles,"[{'iata': 'KL', 'name': 'KLM'}]",8052,605,AMS,Schiphol
39907,Portland,NA,USA,US,"Portland (PDX), USA",30,PDX,KPDX,45.588995,-122.592901,Portland International,"[{'carriers': [{'iata': 'AS', 'name': 'Alaska ...",America/Los_Angeles,"[{'iata': 'AS', 'name': 'Alaska Airlines'}]",2482,220,ANC,Ted Stevens Anchorage International


In [6]:
# Count number of destinations per source airport, add as column "num_dests"
import numpy as np

dests_per_source = master_air_df["iata"].value_counts()
master_air_df["num_dests"] = master_air_df["iata"].map(dests_per_source)
master_air_df[["iata", "num_dests"]].drop_duplicates().sort_values("num_dests", ascending=False)
master_air_df['source_access_index'] = np.select(
    [
        master_air_df["num_dests"] < 10,
        (master_air_df["num_dests"] >= 10) & (master_air_df["num_dests"] < 25),
        (master_air_df["num_dests"] >= 25) & (master_air_df["num_dests"] < 50),
        (master_air_df["num_dests"] >= 50) & (master_air_df["num_dests"] < 100),
        master_air_df["num_dests"] >= 100,
    ],
    [1, 2, 3, 4, 5],
    default=np.nan,
)
master_air_df[master_air_df['iata']=="PDX"].head(3)

,city_name,continent,country,country_code,display_name,elevation,iata,icao,latitude,longitude,name,routes,timezone,carriers,km,min,dest_iata,dest_name,num_dests,source_access_index
39905,Portland,NA,USA,US,"Portland (PDX), USA",30,PDX,KPDX,45.588995,-122.592901,Portland International,"[{'carriers': [{'iata': 'AS', 'name': 'Alaska ...",America/Los_Angeles,"[{'iata': 'AS', 'name': 'Alaska Airlines'}]",1786,183,ABQ,Albuquerque International Sunport,87,4.0
39906,Portland,NA,USA,US,"Portland (PDX), USA",30,PDX,KPDX,45.588995,-122.592901,Portland International,"[{'carriers': [{'iata': 'AS', 'name': 'Alaska ...",America/Los_Angeles,"[{'iata': 'KL', 'name': 'KLM'}]",8052,605,AMS,Schiphol,87,4.0
39907,Portland,NA,USA,US,"Portland (PDX), USA",30,PDX,KPDX,45.588995,-122.592901,Portland International,"[{'carriers': [{'iata': 'AS', 'name': 'Alaska ...",America/Los_Angeles,"[{'iata': 'AS', 'name': 'Alaska Airlines'}]",2482,220,ANC,Ted Stevens Anchorage International,87,4.0


Let's do another self join to get the routes and access index from the sources to the desitnations

In [7]:
access_info = master_air_df[['iata','num_dests','source_access_index']].drop_duplicates()
access_info = access_info.rename(
    columns={
        'iata':'dest_iata',
        'num_dests':'dest_num_dests',
        'source_access_index':'dest_access_index'
    }
)

print(master_air_df.shape)
master_air_df = pd.merge(
    master_air_df,
    access_info,
    on = 'dest_iata',
    how = 'left'
)

print(access_info.shape)
print(master_air_df.shape)

master_air_df[master_air_df['iata']=="PDX"].head(3)

(58342, 20)
(3808, 3)
(58342, 22)


,city_name,continent,country,country_code,display_name,elevation,iata,icao,latitude,longitude,...,timezone,carriers,km,min,dest_iata,dest_name,num_dests,source_access_index,dest_num_dests,dest_access_index
39905,Portland,NA,USA,US,"Portland (PDX), USA",30,PDX,KPDX,45.588995,-122.592901,...,America/Los_Angeles,"[{'iata': 'AS', 'name': 'Alaska Airlines'}]",1786,183,ABQ,Albuquerque International Sunport,87,4.0,31.0,3.0
39906,Portland,NA,USA,US,"Portland (PDX), USA",30,PDX,KPDX,45.588995,-122.592901,...,America/Los_Angeles,"[{'iata': 'KL', 'name': 'KLM'}]",8052,605,AMS,Schiphol,87,4.0,272.0,5.0
39907,Portland,NA,USA,US,"Portland (PDX), USA",30,PDX,KPDX,45.588995,-122.592901,...,America/Los_Angeles,"[{'iata': 'AS', 'name': 'Alaska Airlines'}]",2482,220,ANC,Ted Stevens Anchorage International,87,4.0,50.0,4.0


# Accessibility Metric

In [8]:
import numpy as np

# Source term: for each source airport, (num_dests * source_access_index)
master_air_df['source_term'] = (master_air_df["num_dests"] * master_air_df["source_access_index"])
# Destination sum: for each source, sum over its 87 (or N) routes of (dest_num_dests * dest_access_index)
master_air_df['dest_sum'] = (master_air_df["dest_num_dests"] * master_air_df["dest_access_index"])#.groupby(master_air_df["iata"]).sum()
master_air_df["dest_term"] = master_air_df["iata"].map(master_air_df.groupby("iata")["dest_sum"].sum())# # Normalize to 0-100
master_air_df['access_term_unnorm'] = (np.log1p(master_air_df['source_term']) *0.75) + (np.log1p(master_air_df['dest_term'])*0.25)
master_air_df["access"] = (master_air_df["access_term_unnorm"] - master_air_df["access_term_unnorm"].min()) / (master_air_df["access_term_unnorm"].max() - master_air_df["access_term_unnorm"].min()) * 100

master_air_df[master_air_df['iata']=="IST"].head(10)


,city_name,continent,country,country_code,display_name,elevation,iata,icao,latitude,longitude,...,dest_name,num_dests,source_access_index,dest_num_dests,dest_access_index,source_term,dest_sum,dest_term,access_term_unnorm,access
24583,Istanbul,AS,Turkiye,TR,"Istanbul (IST), Turkiye",325,IST,LTFM,41.260278,28.741944,...,Annaba,316,5.0,11.0,2.0,1580.0,22.0,110615.0,8.427815,100.0
24584,Istanbul,AS,Turkiye,TR,"Istanbul (IST), Turkiye",325,IST,LTFM,41.260278,28.741944,...,Abidjan Felix Houphouet Boigny International,316,5.0,36.0,3.0,1580.0,108.0,110615.0,8.427815,100.0
24585,Istanbul,AS,Turkiye,TR,"Istanbul (IST), Turkiye",325,IST,LTFM,41.260278,28.741944,...,Abuja Nnamdi Azikiwe International,316,5.0,40.0,3.0,1580.0,120.0,110615.0,8.427815,100.0
24586,Istanbul,AS,Turkiye,TR,"Istanbul (IST), Turkiye",325,IST,LTFM,41.260278,28.741944,...,Kotoka International,316,5.0,33.0,3.0,1580.0,99.0,110615.0,8.427815,100.0
24587,Istanbul,AS,Turkiye,TR,"Istanbul (IST), Turkiye",325,IST,LTFM,41.260278,28.741944,...,İzmir Adnan Menderes Airport,316,5.0,95.0,4.0,1580.0,380.0,110615.0,8.427815,100.0
24588,Istanbul,AS,Turkiye,TR,"Istanbul (IST), Turkiye",325,IST,LTFM,41.260278,28.741944,...,Addis Ababa Bole International Airport,316,5.0,128.0,5.0,1580.0,640.0,110615.0,8.427815,100.0
24589,Istanbul,AS,Turkiye,TR,"Istanbul (IST), Turkiye",325,IST,LTFM,41.260278,28.741944,...,Adiyaman Airport,316,5.0,3.0,1.0,1580.0,3.0,110615.0,8.427815,100.0
24590,Istanbul,AS,Turkiye,TR,"Istanbul (IST), Turkiye",325,IST,LTFM,41.260278,28.741944,...,Sochi Airport,316,5.0,74.0,4.0,1580.0,296.0,110615.0,8.427815,100.0
24591,Istanbul,AS,Turkiye,TR,"Istanbul (IST), Turkiye",325,IST,LTFM,41.260278,28.741944,...,Málaga Airport,316,5.0,162.0,5.0,1580.0,810.0,110615.0,8.427815,100.0
24592,Istanbul,AS,Turkiye,TR,"Istanbul (IST), Turkiye",325,IST,LTFM,41.260278,28.741944,...,Agri,316,5.0,4.0,1.0,1580.0,4.0,110615.0,8.427815,100.0


In [9]:
what_did_i_do = master_air_df[['iata','name','num_dests','access']]
what_did_i_do = what_did_i_do.sort_values('access', ascending=False).drop_duplicates()
what_did_i_do.head(25)

,iata,name,num_dests,access
24707,IST,Istanbul Airport,316,100.000000
18804,FRA,Frankfurt Airport,286,99.358711
9358,CDG,Charles De Gaulle,271,98.829420
2177,AMS,Schiphol,272,98.751651
15870,DXB,Dubai International Airport,268,98.178401
38671,ORD,Chicago O Hare International,279,97.478580
17821,FCO,Fiumicino,241,97.467600
32129,MAD,Barajas,232,96.877987
14116,DFW,Dallas Fort Worth International,272,96.851197
35346,MUC,Munich Airport,229,96.799725
